# Insurance Claim Risk & Fraud Prediction System
## Exploratory Data Analysis & Model Development
**Enterprise Marsh Analytics Standard**

This notebook provides end-to-end data exploration, actuarial feature engineering lift analysis, and model benchmark evaluations for insurance claims fraud detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

### 1. Load Raw Claims Dataset & Summary Statistics
Loading 20,000+ synthesized claims records.

In [ ]:
data_path = Path('../data/raw/insurance_claims_raw.csv')
if not data_path.exists():
    data_path = Path('data/raw/insurance_claims_raw.csv')

df = pd.read_csv(data_path)
print(f'Total Claims: {len(df):,}')
print(f'Features: {df.shape[1]}')
print(f'Fraud Rate: {df["fraud_flag"].mean():.2%}')
df.head()

### 2. Class Imbalance Analysis
Insurance fraud is naturally imbalanced. In real-world actuarial portfolios, fraudulent claims represent between 8% and 15% of all notices of loss.

In [ ]:
counts = df['fraud_flag'].value_counts()
labels = ['Legitimate (0)', 'Fraudulent (1)']
colors = ['#10B981', '#EF4444']

plt.figure(figsize=(7, 4))
bars = plt.bar(labels, counts.values, color=colors, width=0.5)
plt.title('Target Distribution: Legitimate vs Fraudulent Claims', fontsize=13, fontweight='bold')
plt.ylabel('Number of Claims')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 200, f'{yval:,} ({yval/len(df):.1%})', ha='center', fontweight='bold')
plt.show()

### 3. Claim Amount Distribution by Fraud Status
**Observation**: Fraudulent claims exhibit a substantially fatter right-tail distribution, reflecting opportunistic and syndicate-driven payout maximization.

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df[df['fraud_flag']==0]['claim_amount'], label='Legitimate', color='#10B981', fill=True, alpha=0.3)
sns.kdeplot(data=df[df['fraud_flag']==1]['claim_amount'], label='Fraudulent', color='#EF4444', fill=True, alpha=0.3)
plt.title('Claim Amount Distribution (Fraud vs Legitimate)', fontsize=13, fontweight='bold')
plt.xlabel('Claim Amount ($)')
plt.ylabel('Density')
plt.legend()
plt.show()

### 4. Claim Delay vs Fraud Probability
**Observation**: Reporting delay is one of the strongest behavioral indicators of staged or fabricated claims. Claims reported >15 days after the incident show more than double the baseline fraud probability.

In [ ]:
delay_bins = pd.cut(df['claim_delay_days'], bins=[-1, 2, 7, 14, 30, 120], labels=['0-2d', '3-7d', '8-14d', '15-30d', '>30d'])
delay_rates = df.groupby(delay_bins, observed=False)['fraud_flag'].mean() * 100

plt.figure(figsize=(9, 4.5))
bars = plt.bar(delay_rates.index.astype(str), delay_rates.values, color='#F59E0B', edgecolor='#D97706', width=0.55)
plt.title('Fraud Rate (%) by Claim Reporting Delay Interval', fontsize=13, fontweight='bold')
plt.xlabel('Reporting Delay Window')
plt.ylabel('Fraud Rate (%)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.6, f'{yval:.1f}%', ha='center', fontweight='bold')
plt.show()

### 5. Accident Severity & Police Report Cross-Tabulation
**Observation**: Unwitnessed, severe accidents without official police documentation exhibit critical risk concentration.

In [ ]:
pivot = df.pivot_table(index='accident_severity', columns='police_report', values='fraud_flag', aggfunc='mean') * 100
plt.figure(figsize=(8, 4.5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Fraud Rate (%)'})
plt.title('Fraud Rate (%) across Severity vs Police Documentation', fontsize=13, fontweight='bold')
plt.xlabel('Police Report Filed?')
plt.ylabel('Accident Severity')
plt.show()

### 6. Production Model Performance & Metadata
Loading serialized production metrics generated by the 5-fold cross-validation and hyperparameter tuning pipeline.

In [ ]:
meta_path = Path('../models/model_metadata.json')
if not meta_path.exists():
    meta_path = Path('models/model_metadata.json')

with open(meta_path, 'r') as f:
    meta = json.load(f)

print(f"Best Model: {meta['best_model_architecture']}")
print(f"Operational Threshold: {meta['decision_thresholds']['operational_selected']}")
print(f"Feature Engineering Lift: +{meta['feature_engineering_lift']['f1_improvement_pct']}% F1 lift")
pd.DataFrame(meta['model_comparison']).T